In [ ]:
from flask import Flask, render_template, request, redirect, url_for, flash
import os
import numpy as np
import tensorflow as tf
from werkzeug.utils import secure_filename
from PIL import Image
import base64
import io

app = Flask(__name__)
app.secret_key = 'your_secret_key_here'

# Configuration for file uploads
UPLOAD_FOLDER = 'static/uploads'
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif'}
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# Load the trained pneumonia detection model
model = tf.keras.models.load_model("pneumonia_detection_model_densenet.h5")
class_names = ["Normal", "Pneumonia"]

def allowed_file(filename):
    """Check if uploaded file has an allowed extension."""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route('/')
def home():
    """Render homepage."""
    return render_template('index.html')

@app.route('/detect', methods=['GET', 'POST'])
def detect():
    """Handle image uploads and make pneumonia predictions."""
    if request.method == 'POST':
        if 'file' not in request.files:
            flash('No file part')
            return redirect(request.url)

        file = request.files['file']

        if file.filename == '':
            flash('No selected file')
            return redirect(request.url)

        if file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
            file.save(filepath)

            # Predict pneumonia using the model
            result = predict_pneumonia(filepath)

            return render_template('detect.html', 
                                   uploaded_image=filename,
                                   result=result)
    
    return render_template('detect.html')

def predict_pneumonia(image_path):
    """
    Process the uploaded image and make a prediction using the model.
    Returns a dictionary with the result and confidence score.
    """
    try:
        # Load and preprocess image
        img = Image.open(image_path).convert('RGB')  # Ensure 3-channel image
        img = img.resize((224, 224))  # Resize to match model input
        img_array = np.array(img) / 255.0  # Normalize pixel values
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

        # Predict using the model
        prediction = model.predict(img_array)
        predicted_class = np.argmax(prediction, axis=1)[0]  
        confidence = np.max(prediction)  


        return {
            'class': class_names[predicted_class],  # Fix: Add predicted class
            'confidence': confidence  # Confidence score
        }
    
    except Exception as e:
        return {'error': str(e)}

@app.route('/about')
def about():
    """Render about page."""
    return render_template('about.html')

if __name__ == '__main__':
    os.makedirs(UPLOAD_FOLDER, exist_ok=True)  # Ensure upload folder exists
    app.run(debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Mar/2025 04:11:18] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:18] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:18] "GET /static/js/main.js HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:20] "GET /detect HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:20] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:20] "GET /static/js/main.js HTTP/1.1" 304 -


1/1 [==============================] - 1s 1s/step


127.0.0.1 - - [27/Mar/2025 04:11:28] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:28] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:28] "GET /static/uploads/person59_virus_116.jpeg HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:28] "GET /static/js/main.js HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:46] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:46] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:46] "GET /static/js/main.js HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:49] "GET /about HTTP/1.1" 200 -
127.0.0.1 - - [27/Mar/2025 04:11:49] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Mar/2025 04:11:49] "GET /static/js/main.js HTTP/1.1" 304 -
